# 06 — Label probabilities and target-language instructions

Two analyses of the neutral collapse on Romanized Hindi and Arabic:

* **Label probabilities.** Re-runs the original inference and records the probability the model assigns to each
  label word at the first output step, so that per-class AUC and a balanced decision rule can be computed
  (`src/label_prob_analysis.py`). This tests whether the collapse is a bias towards the neutral label or a loss
  of class separation.
* **Target-language instruction.** Repeats the evaluation of the non-English test sets with the instruction
  written in the language of the input (label words kept in English).

**Runtime:** Colab, **T4 GPU** (Runtime → Change runtime type → T4). About 45–60 min in total.

**Setup reproduced:** the loader of `src/infer_local.py` (Unsloth, 4-bit), the same chat template, batching
(batch size 8, original item order), greedy decoding and label parser. Cell 4 reports agreement with the released
cached predictions in `results/api_cache/`.

**Output:** `results/label_probs/` and `results/target_language_prompts/`, packaged as `label_prob_outputs.zip`
(downloaded at the end, and saved to Google Drive if mounted). Unzip into the repo root.


In [ ]:
# 1. Setup: clone the repository (src/ and the cached predictions) and install dependencies.
#    Safe to re-run. Colab already has numpy, pandas, scikit-learn, matplotlib and seaborn.
#    PIN_VERSIONS: set to True if cell 4 reports agreement well below 1.00; it installs the versions used
#    for the paper (Unsloth 2026.4.4, Transformers 5.5.0). Then Runtime -> Restart session and run again.
PIN_VERSIONS = False
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.isdir('/content/repo'):
    !git clone -q https://github.com/Mrigank005/slm-cross-lingual-transfer.git /content/repo
%cd /content/repo
if PIN_VERSIONS:
    %pip install -q "unsloth==2026.4.4" "transformers==5.5.0"
else:
    %pip install -q unsloth
# Versions used for the paper: unsloth 2026.4.4, transformers 5.5.0, torch 2.10.0+cu128
!pip show unsloth transformers torch 2>/dev/null | grep -E '^(Name|Version)'

In [ ]:
# 2. Optional: mount Drive so outputs survive a disconnect. Skip if you prefer the zip download only.
USE_DRIVE = True
OUT_ROOT = '/content/out'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = '/content/drive/MyDrive/slm_label_probs'
import os; os.makedirs(OUT_ROOT, exist_ok=True); print('Outputs ->', OUT_ROOT)

In [ ]:
# 3. Shared code: model list, original-order data, and a generate() that also returns first-token label log-probs.
import os, sys, json, time, gc
os.chdir('/content/repo'); sys.path.insert(0, '/content/repo')  # survives a runtime restart
assert 'OUT_ROOT' in globals(), 'Run cell 2 first (after any restart, re-run cells 2 and 3)'
from pathlib import Path
import numpy as np, torch
assert torch.cuda.is_available(), 'Switch the runtime to a T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

import unsloth  # must be imported before transformers-based loaders
from src.infer_local import load_base_model, load_finetuned_model
from src.prompts import SYSTEM_PROMPT, build_user_prompt
from src.evaluate import normalize_label

LANGS = ['english', 'spanish', 'hindi', 'arabic']
MODELS = {  # name -> (loader, repo, cached-predictions dir)
    'qwen25_base':          (load_base_model,      'unsloth/Qwen2.5-3B-Instruct',                 'results/api_cache/qwen25_base/{lang}'),
    'qwen25_finetuned_s42': (load_finetuned_model, 'Mrigank005/SLM-sentiment-crosslingual-seed-42',  'results/api_cache/qwen25_finetuned/seed_42/{lang}'),
    'qwen25_finetuned_s123':(load_finetuned_model, 'Mrigank005/SLM-sentiment-crosslingual-seed-123', 'results/api_cache/qwen25_finetuned/seed_123/{lang}'),
    'qwen25_finetuned_s456':(load_finetuned_model, 'Mrigank005/SLM-sentiment-crosslingual-seed-456', 'results/api_cache/qwen25_finetuned/seed_456/{lang}'),
}

def load_cached(dirpath):
    d = json.load(open(Path(dirpath) / 'predictions.json', encoding='utf-8'))['predictions']
    assert len(d) == 870, (dirpath, len(d))
    return d  # insertion order = original processing order

def label_token_groups(tok):
    # class index -> token ids whose first-token probability counts for that class
    groups = {}
    for c, w in enumerate(['negative', 'neutral', 'positive']):
        ids = set()
        for v in (w, w.capitalize(), ' ' + w, ' ' + w.capitalize()):
            ids.add(tok.encode(v, add_special_tokens=False)[0])
        groups[c] = sorted(ids)
    return groups

@torch.no_grad()
def run_batched(model, tok, texts, system_prompt, user_fn, batch_size=8):
    # Mirrors src/infer_local.predict_batch, plus first-step scores.
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token; tok.pad_token_id = tok.eos_token_id
    groups = label_token_groups(tok)
    raws, logps, top_is_label = [], [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        prompts = [tok.apply_chat_template([{'role': 'system', 'content': system_prompt},
                                            {'role': 'user', 'content': user_fn(t)}],
                                           tokenize=False, add_generation_prompt=True) for t in batch]
        enc = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device)
        # Label generated exactly as in the original run.
        seqs = model.generate(**enc, max_new_tokens=10, temperature=0.0, do_sample=False,
                              pad_token_id=tok.pad_token_id)
        # First-token distribution from a plain forward pass. The tokenizer pads on the left,
        # so position -1 is the last prompt token for every row. This is used instead of
        # generate(output_scores=True), which Unsloth's patched generate may not return.
        assert tok.padding_side == 'left', ('Tokenizer pads on the', tok.padding_side,
                                            '- environment differs from the original setup')
        logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask).logits[:, -1, :]
        first = torch.log_softmax(logits.float(), dim=-1)  # [B, V]
        allids = [i for g in groups.values() for i in g]
        for j in range(len(batch)):
            gen = seqs[j][enc.input_ids.shape[1]:]
            raws.append(tok.decode(gen, skip_special_tokens=True).strip())
            logps.append([float(torch.logsumexp(first[j, groups[c]], 0)) for c in range(3)])
            top_is_label.append(int(first[j].argmax().item()) in allids)
    return raws, np.array(logps), np.array(top_is_label)

def save(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp'); tmp.write_text(json.dumps(obj, ensure_ascii=False)); tmp.replace(path)
print('ready')

In [ ]:
# 4. Label probabilities: reproduce the original runs (English instruction) and record label log-probs.
#    Resumable: finished model x language files are skipped.
lp_dir = Path(OUT_ROOT) / 'results/label_probs'
for name, (loader, repo, cdir) in MODELS.items():
    todo = [l for l in LANGS if not (lp_dir / f'{name}_{l}.json').exists()]
    if not todo:
        print(name, 'done'); continue
    model, tok = loader(repo)
    for lang in todo:
        cached = load_cached(cdir.format(lang=lang))
        ids = list(cached); texts = [cached[k]['text'] for k in ids]
        t0 = time.time()
        raws, logp, top_ok = run_batched(model, tok, texts, SYSTEM_PROMPT, build_user_prompt)
        pred = [normalize_label(r) for r in raws]
        old = [cached[k]['pred_label'] for k in ids]
        match = float(np.mean(np.array(pred) == np.array(old)))
        save(lp_dir / f'{name}_{lang}.json', {
            'model': name, 'repo': repo, 'language': lang, 'prompt': 'english',
            'ids': ids, 'true_label': [cached[k]['true_label'] for k in ids],
            'cached_pred': old, 'new_pred': pred, 'raw_output': raws,
            'logp_neg_neu_pos': logp.tolist(), 'top_token_is_label': top_ok.tolist(),
            'match_rate_vs_cache': match, 'seconds': time.time() - t0})
        print(f'{name:22s} {lang:8s} agreement with cached predictions = {match:.4f}  top-token-is-label = {top_ok.mean():.3f}')
    del model, tok; gc.collect(); torch.cuda.empty_cache()

### Target-language instructions

The system and user messages are translated; the **label words stay in English** (`positive`, `neutral`,
`negative`) so that outputs are parsed and scored exactly as with the English instruction.


In [ ]:
# 5. Target-language instructions (label words kept in English).
TARGET_PROMPTS = {
    'spanish': (
        'Eres un clasificador de sentimiento. Clasifica el sentimiento del texto dado como exactamente una de '
        'estas etiquetas: positive, neutral o negative. Responde SOLO con la etiqueta de sentimiento, nada más.',
        lambda t: 'Clasifica el sentimiento del siguiente texto como positive, neutral o negative.\n\nTexto: ' + t.strip()),
    'hindi': (
        'Tum ek sentiment classifier ho. Diye gaye text ke sentiment ko in teen mein se sirf ek label do: '
        'positive, neutral, ya negative. Sirf sentiment label likho, aur kuch nahi.',
        lambda t: 'Neeche diye gaye text ka sentiment positive, neutral, ya negative mein batao.\n\nText: ' + t.strip()),
    'arabic': (
        'أنت مصنِّف للمشاعر. صنِّف مشاعر النص المعطى بواحدة فقط من هذه التسميات: positive أو neutral أو negative. '
        'أجب بتسمية المشاعر فقط، ولا شيء غير ذلك.',
        lambda t: 'صنِّف مشاعر النص التالي إلى positive أو neutral أو negative.\n\nالنص: ' + t.strip()),
}
for k, (s, u) in TARGET_PROMPTS.items():
    print(f'--- {k}\nSYSTEM: {s}\nUSER:   {u("<tweet>")}\n')

In [ ]:
# 6. Target-language instruction: same models, non-English test sets only. Resumable.
tp_dir = Path(OUT_ROOT) / 'results/target_language_prompts'
for name, (loader, repo, cdir) in MODELS.items():
    todo = [l for l in TARGET_PROMPTS if not (tp_dir / f'{name}_{l}.json').exists()]
    if not todo:
        print(name, 'done'); continue
    model, tok = loader(repo)
    for lang in todo:
        cached = load_cached(cdir.format(lang=lang))
        ids = list(cached); texts = [cached[k]['text'] for k in ids]
        sys_p, user_fn = TARGET_PROMPTS[lang]
        raws, logp, top_ok = run_batched(model, tok, texts, sys_p, user_fn)
        pred = [normalize_label(r) for r in raws]
        save(tp_dir / f'{name}_{lang}.json', {
            'model': name, 'repo': repo, 'language': lang, 'prompt': 'target_language',
            'system_prompt': sys_p, 'user_template': user_fn('{text}'),
            'ids': ids, 'true_label': [cached[k]['true_label'] for k in ids],
            'new_pred': pred, 'raw_output': raws, 'logp_neg_neu_pos': logp.tolist(),
            'top_token_is_label': top_ok.tolist()})
        neu = float(np.mean(np.array(pred) == 1))
        print(f'{name:22s} {lang:8s} neutral share (target-language instruction) = {neu:.3f}')
    del model, tok; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 7. Package everything for download.
import shutil
zip_base = '/content/label_prob_outputs'
shutil.make_archive(zip_base, 'zip', OUT_ROOT, 'results')
from google.colab import files
files.download(zip_base + '.zip')